In [ ]:
import os, random, time
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================

# File CSV mới upload lên Colab
CSV_PATH = "DatasetMTH_coeffs_DT123_5levels.csv"

# Input variables
VAR_COLS = [f"var{i}" for i in range(1, 11)]

# Output columns
FCR_COL = "Fcr"

# File mới dùng Lambda1, Lambda2 thay vì L1, L2
L1_COL = "Lambda1"
L2_COL = "Lambda2"

# Lambda0 có trong file nhưng về bản chất trùng với Fcr,
# nên không cần dùng làm output riêng
L0_COL = "Lambda0"

# W grid for curve reconstruction (fixed, not an input)
Nw = 50
W_GRID = np.linspace(0.0, 1.0, Nw).astype(np.float32)


# Model
ENC = [256,256,256]       # encoder widths; zdim = last element
ACT = "relu"          # NOTE: lower-case for get_act
DROPOUT = 0.0
BATCHNORM = False

# Training
EPOCHS = 200
BATCH_SIZE = 4096
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Loss weights (multitask)
LAMBDA_FCR   = 2.5
LAMBDA_SHAPE = 1.0
USE_CURVE_LOSS = False
LAMBDA_CURVE = 1.0

# Scheduler / Early stop
LR_FACTOR = 0.5
LR_PATIENCE = 4
LR_MIN = 1e-6

ES_PATIENCE = 8
ES_MIN_DELTA = 1e-5
VERBOSE_EVERY = 1

# Mixed precision (AMP)
USE_AMP = True

# =========================
# UTILS
# =========================
def seed_everything(seed=42, deterministic=False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
        try:
            torch.use_deterministic_algorithms(False)
        except Exception:
            pass

def get_act(name: str):
    name = name.lower()
    table = {
        "relu": nn.ReLU(),
        "gelu": nn.GELU(),
        "silu": nn.SiLU(),
        "tanh": nn.Tanh(),
    }
    if name not in table:
        raise ValueError(f"Unsupported activation: {name}. Choose from {list(table.keys())}")
    return table[name]

def mlp(in_dim, widths, out_dim, act="silu", dropout=0.0, batchnorm=False):
    a = get_act(act)
    layers, d = [], in_dim
    for w in widths:
        layers.append(nn.Linear(d, w))
        if batchnorm:
            layers.append(nn.BatchNorm1d(w))
        layers.append(a)
        if dropout and dropout > 0:
            layers.append(nn.Dropout(dropout))
        d = w
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

@torch.no_grad()
def measure_inference_time(model, input_dim=10, device="cpu", batch_size=1,
                           n_warmup=50, n_iters=200, use_amp=True):
    """
    Forward-only inference time for: model(x) -> (fcr, shp, curve).
    Returns (ms_per_batch, ms_per_sample, samples_per_second).
    """
    model.eval()
    is_cuda = device.startswith("cuda")
    amp_on = (use_amp and is_cuda)

    xb = torch.randn(batch_size, input_dim, device=device)

    # warmup
    for _ in range(n_warmup):
        with torch.cuda.amp.autocast(enabled=amp_on):
            _ = model(xb)
    if is_cuda:
        torch.cuda.synchronize()

    # timed
    t0 = time.perf_counter()
    for _ in range(n_iters):
        with torch.cuda.amp.autocast(enabled=amp_on):
            _ = model(xb)
    if is_cuda:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    total_s = t1 - t0
    ms_per_batch = (total_s / n_iters) * 1e3
    ms_per_sample = ms_per_batch / batch_size
    samples_per_s = (batch_size * n_iters) / total_s
    return ms_per_batch, ms_per_sample, samples_per_s

# =========================
# MODEL: Level + Shape (no redundancy)
# =========================
class LevelShapeParabolaNet(nn.Module):
    """
    Shared encoder: X(10) -> z
    Head A: z -> Fcr (scalar)  [level term, equals L0]
    Head B: z -> (L1, L2)      [shape-only]
    Hard constraint: L0 := Fcr
    Curve: lam_hat(W) = Fcr + L1*W + L2*W^2 using fixed basis Phi
    """
    def __init__(self, enc, W_grid, act="silu", dropout=0.0, batchnorm=False):
        super().__init__()
        assert len(enc) >= 1
        zdim = enc[-1]
        enc_hidden = enc[:-1]

        self.encoder  = mlp(10, enc_hidden, zdim, act=act, dropout=dropout, batchnorm=batchnorm)
        self.head_fcr = nn.Linear(zdim, 1)
        self.head_shp = nn.Linear(zdim, 2)

        W = torch.as_tensor(W_grid, dtype=torch.float32)
        Phi = torch.stack([torch.ones_like(W), W, W**2], dim=0)  # (3, Nw)
        self.register_buffer("Phi", Phi)

    def forward(self, x):
        z = self.encoder(x)
        fcr = self.head_fcr(z)      # (B,1)
        shp = self.head_shp(z)      # (B,2) => (L1,L2)

        L0 = fcr
        L1 = shp[:, 0:1]
        L2 = shp[:, 1:2]
        L  = torch.cat([L0, L1, L2], dim=1)     # (B,3)

        curve = L @ self.Phi                    # (B,Nw)
        return fcr, shp, curve

# =========================
# DATASET / LOADER
# =========================
class MultiTaskDataset(Dataset):
    def __init__(self, X, Fcr, Lshape):
        self.X = torch.tensor(X, dtype=torch.float32)                      # (N,10)
        self.F = torch.tensor(Fcr, dtype=torch.float32).reshape(-1, 1)     # (N,1)
        self.S = torch.tensor(Lshape, dtype=torch.float32)                 # (N,2)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.F[i], self.S[i]

def make_loader(X, Fcr, Lshape, bs, shuffle, pin_memory):
    return DataLoader(
        MultiTaskDataset(X, Fcr, Lshape),
        batch_size=bs,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=pin_memory
    )

# =========================
# TRAIN / EVAL
# =========================
def fit(model, dl_tr, dl_va, device="cpu"):
    model.to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    mse = nn.MSELoss()
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=LR_MIN
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.startswith("cuda")))
    best, best_state, bad = float("inf"), None, 0
    t0 = time.perf_counter()

    history = {
        "epoch": [],
        "lr": [],
        "tr_total": [], "va_total": [],
        "tr_fcr": [],   "va_fcr":   [],
        "tr_shp": [],   "va_shp":   [],
        "tr_cur": [],   "va_cur":   [],
    }

    for ep in range(1, EPOCHS + 1):
        # ---- train ----
        model.train()
        tr_tot = tr_f = tr_s = tr_c = 0.0
        ntr = 0

        for xb, fb, sb in dl_tr:
            xb, fb, sb = xb.to(device), fb.to(device), sb.to(device)

            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.startswith("cuda"))):
                fcr_p, shp_p, curve_p = model(xb)
                loss_f = mse(fcr_p, fb)
                loss_s = mse(shp_p, sb)

                if USE_CURVE_LOSS:
                    L_true = torch.cat([fb, sb[:, 0:1], sb[:, 1:2]], dim=1)  # (B,3)
                    curve_true = L_true @ model.Phi
                    loss_c = mse(curve_p, curve_true)
                else:
                    loss_c = 0.0

                loss = LAMBDA_FCR * loss_f + LAMBDA_SHAPE * loss_s + (LAMBDA_CURVE * loss_c if USE_CURVE_LOSS else 0.0)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            bs = xb.size(0)
            tr_tot += float(loss) * bs
            tr_f   += float(loss_f) * bs
            tr_s   += float(loss_s) * bs
            tr_c   += (float(loss_c) * bs) if USE_CURVE_LOSS else 0.0
            ntr += bs

        tr_total = tr_tot / ntr
        tr_f_m   = tr_f   / ntr
        tr_s_m   = tr_s   / ntr
        tr_c_m   = tr_c   / ntr if USE_CURVE_LOSS else 0.0

        # ---- val ----
        model.eval()
        va_tot = va_f = va_s = va_c = 0.0
        nva = 0
        with torch.no_grad():
            for xb, fb, sb in dl_va:
                xb, fb, sb = xb.to(device), fb.to(device), sb.to(device)
                fcr_p, shp_p, curve_p = model(xb)

                loss_f = mse(fcr_p, fb)
                loss_s = mse(shp_p, sb)

                if USE_CURVE_LOSS:
                    L_true = torch.cat([fb, sb[:, 0:1], sb[:, 1:2]], dim=1)
                    curve_true = L_true @ model.Phi
                    loss_c = mse(curve_p, curve_true)
                else:
                    loss_c = 0.0

                loss = LAMBDA_FCR * loss_f + LAMBDA_SHAPE * loss_s + (LAMBDA_CURVE * loss_c if USE_CURVE_LOSS else 0.0)

                bs = xb.size(0)
                va_tot += float(loss) * bs
                va_f   += float(loss_f) * bs
                va_s   += float(loss_s) * bs
                va_c   += (float(loss_c) * bs) if USE_CURVE_LOSS else 0.0
                nva += bs

        va_total = va_tot / nva
        va_f_m   = va_f   / nva
        va_s_m   = va_s   / nva
        va_c_m   = va_c   / nva if USE_CURVE_LOSS else 0.0

        sch.step(va_total)
        cur_lr = opt.param_groups[0]["lr"]

        improved = (best - va_total) > ES_MIN_DELTA
        if improved:
            best = va_total
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        history["epoch"].append(ep)
        history["lr"].append(cur_lr)
        history["tr_total"].append(tr_total); history["va_total"].append(va_total)
        history["tr_fcr"].append(tr_f_m);     history["va_fcr"].append(va_f_m)
        history["tr_shp"].append(tr_s_m);     history["va_shp"].append(va_s_m)
        history["tr_cur"].append(tr_c_m);     history["va_cur"].append(va_c_m)

        if VERBOSE_EVERY and (ep == 1 or ep % VERBOSE_EVERY == 0):
            print(
                f"Ep {ep:03d} | "
                f"tr={tr_total:.6f} (fcr={tr_f_m:.6f}, shp={tr_s_m:.6f}" + (f", cur={tr_c_m:.6f}" if USE_CURVE_LOSS else "") + ") | "
                f"va={va_total:.6f} (fcr={va_f_m:.6f}, shp={va_s_m:.6f}" + (f", cur={va_c_m:.6f}" if USE_CURVE_LOSS else "") + ") | "
                f"lr={cur_lr:.2e} | best={best:.6f} | bad={bad}/{ES_PATIENCE}"
            )

        if bad >= ES_PATIENCE:
            print(f"Early stop @ ep {ep}, best val_total={best:.6f}")
            break

    ttrain = time.perf_counter() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, ttrain, history

@torch.no_grad()
def eval_original_scale(model, dl, scF: StandardScaler, scS: StandardScaler, device="cpu"):
    model.eval()
    f_pred_s, f_true_s = [], []
    s_pred_s, s_true_s = [], []

    for xb, fb, sb in dl:
        xb = xb.to(device)
        fb = fb.to(device)
        sb = sb.to(device)

        fcr_p_s, shp_p_s, _ = model(xb)  # scaled outputs
        f_pred_s.append(fcr_p_s.cpu().numpy())
        f_true_s.append(fb.cpu().numpy())
        s_pred_s.append(shp_p_s.cpu().numpy())
        s_true_s.append(sb.cpu().numpy())

    f_pred_s = np.vstack(f_pred_s); f_true_s = np.vstack(f_true_s)
    s_pred_s = np.vstack(s_pred_s); s_true_s = np.vstack(s_true_s)

    f_pred = scF.inverse_transform(f_pred_s).reshape(-1, 1)
    f_true = scF.inverse_transform(f_true_s).reshape(-1, 1)

    s_pred = scS.inverse_transform(s_pred_s)
    s_true = scS.inverse_transform(s_true_s)

    rmse_f = float(np.sqrt(np.mean((f_pred - f_true) ** 2)))
    rmse_s = float(np.sqrt(np.mean((s_pred - s_true) ** 2)))

    W = W_GRID.reshape(1, -1)  # (1,Nw)
    curve_pred = f_pred + s_pred[:, 0:1] * W + s_pred[:, 1:2] * (W**2)
    curve_true = f_true + s_true[:, 0:1] * W + s_true[:, 1:2] * (W**2)
    rmse_curve = float(np.sqrt(np.mean((curve_pred - curve_true) ** 2)))

    return rmse_f, rmse_s, rmse_curve

# =========================
# RUN
# =========================
# =========================
# RUN
# =========================
seed_everything(42, deterministic=False)

# Đọc file CSV.
# Nếu chạy Colab và file chưa có sẵn trong /content,
# đoạn này sẽ mở hộp upload file.
try:
    from google.colab import files

    if not os.path.exists(CSV_PATH):
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]

except Exception:
    # Nếu không chạy Colab, dùng trực tiếp CSV_PATH ở local
    pass

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

print("CSV_PATH:", CSV_PATH)
print("CSV shape:", df.shape)
print("Columns:", df.columns.tolist())

# Check các cột bắt buộc
required_cols = VAR_COLS + [FCR_COL, L1_COL, L2_COL]
missing = [c for c in required_cols if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing columns in CSV: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

# Inputs
X = df[VAR_COLS].values.astype(np.float32)

# Outputs
Fcr = df[FCR_COL].astype(np.float32).values.reshape(-1, 1)
L1  = df[L1_COL].astype(np.float32).values.reshape(-1, 1)
L2  = df[L2_COL].astype(np.float32).values.reshape(-1, 1)

Lshape = np.concatenate([L1, L2], axis=1)

# Optional sanity check: Lambda0 nên gần như trùng Fcr
if L0_COL in df.columns:
    max_diff_L0_Fcr = np.max(np.abs(df[L0_COL].values - df[FCR_COL].values))
    print(f"Max |Lambda0 - Fcr| = {max_diff_L0_Fcr:.6e}")

# split: train/test then train/val
X_tr, X_te, F_tr, F_te, S_tr, S_te = train_test_split(
    X, Fcr, Lshape,
    test_size=0.2,
    random_state=42
)

X_tr, X_va, F_tr, F_va, S_tr, S_va = train_test_split(
    X_tr, F_tr, S_tr,
    test_size=0.1,
    random_state=42
)

# scale train-only
scX, scF, scS = StandardScaler(), StandardScaler(), StandardScaler()

X_tr_s = scX.fit_transform(X_tr)
X_va_s = scX.transform(X_va)
X_te_s = scX.transform(X_te)

F_tr_s = scF.fit_transform(F_tr)
F_va_s = scF.transform(F_va)
F_te_s = scF.transform(F_te)

S_tr_s = scS.fit_transform(S_tr)
S_va_s = scS.transform(S_va)
S_te_s = scS.transform(S_te)

device = "cuda" if torch.cuda.is_available() else "cpu"
pin_memory = device.startswith("cuda")

dl_tr = make_loader(
    X_tr_s, F_tr_s, S_tr_s,
    BATCH_SIZE,
    shuffle=True,
    pin_memory=pin_memory
)

dl_va = make_loader(
    X_va_s, F_va_s, S_va_s,
    BATCH_SIZE,
    shuffle=False,
    pin_memory=pin_memory
)

dl_te = make_loader(
    X_te_s, F_te_s, S_te_s,
    BATCH_SIZE,
    shuffle=False,
    pin_memory=pin_memory
)

model = LevelShapeParabolaNet(
    ENC,
    W_grid=W_GRID,
    act=ACT,
    dropout=DROPOUT,
    batchnorm=BATCHNORM
).to(device)

p_all, p_tr = count_params(model)

print(f"Params: total={p_all:,} | trainable={p_tr:,} | device={device}")
print(f"n_train={len(X_tr_s):,} | n_val={len(X_va_s):,} | n_test={len(X_te_s):,}")
print(
    f"Loss weights: fcr={LAMBDA_FCR}, "
    f"shape={LAMBDA_SHAPE}, "
    f"curve={LAMBDA_CURVE} "
    f"(USE_CURVE_LOSS={USE_CURVE_LOSS})"
)
print("Model: X->z; z->Fcr; z->(Lambda1,Lambda2); L0:=Fcr; curve=Fcr+Lambda1*W+Lambda2*W^2")

model, ttrain, history = fit(model, dl_tr, dl_va, device=device)

rmse_f_val, rmse_s_val, rmse_c_val = eval_original_scale(model, dl_va, scF, scS, device=device)
rmse_f_te,  rmse_s_te,  rmse_c_te  = eval_original_scale(model, dl_te, scF, scS, device=device)

print(f"Train time: {ttrain:.3f}s")
print(f"VAL : RMSE_Fcr={rmse_f_val:.6f} | RMSE_(L1,L2)={rmse_s_val:.6f} | RMSE_Curve50={rmse_c_val:.6f}")
print(f"TEST: RMSE_Fcr={rmse_f_te :.6f} | RMSE_(L1,L2)={rmse_s_te :.6f} | RMSE_Curve50={rmse_c_te :.6f}")

@torch.no_grad()
def predict_time_full_test(model, dl, device="cpu", use_amp=False, n_warmup_batches=5):
    model.eval()
    is_cuda = str(device).startswith("cuda")
    amp_on = (use_amp and is_cuda)

    # warmup
    it = iter(dl)
    for _ in range(n_warmup_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        xb = batch[0].to(device, non_blocking=True)  # batch[0] luôn là X
        if amp_on:
            with torch.cuda.amp.autocast(True):
                _ = model(xb)
        else:
            _ = model(xb)
    if is_cuda:
        torch.cuda.synchronize()

    # timed full pass
    n = 0
    t0 = time.perf_counter()
    for batch in dl:
        xb = batch[0].to(device, non_blocking=True)
        n += xb.size(0)
        if amp_on:
            with torch.cuda.amp.autocast(True):
                _ = model(xb)
        else:
            _ = model(xb)
    if is_cuda:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    total_s = t1 - t0
    ms_per_sample = (total_s / max(n,1)) * 1e3
    sps = n / max(total_s, 1e-12)
    return total_s, ms_per_sample, sps, n

total_s, ms_s, sps, n = predict_time_full_test(model, dl_te, device=device, use_amp=False, n_warmup_batches=5)
print(f"FULL TEST predict time (forward-only): total={total_s:.4f}s | {ms_s:.6f} ms/sample | {sps:.1f} samples/s | n={n:,} | device={device}")




CSV_PATH: DatasetMTH_coeffs_DT123_5levels.csv
CSV shape: (2343750, 14)
Columns: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'var10', 'Fcr', 'Lambda0', 'Lambda1', 'Lambda2']
Max |Lambda0 - Fcr| = 0.000000e+00
Params: total=135,171 | trainable=135,171 | device=cuda
n_train=1,687,500 | n_val=187,500 | n_test=468,750
Loss weights: fcr=2.5, shape=1.0, curve=1.0 (USE_CURVE_LOSS=False)
Model: X->z; z->Fcr; z->(Lambda1,Lambda2); L0:=Fcr; curve=Fcr+Lambda1*W+Lambda2*W^2


/tmp/ipykernel_558/1358062619.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.startswith("cuda")))
/tmp/ipykernel_558/1358062619.py:259: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(USE_AMP and device.startswith("cuda"))):
/tmp/ipykernel_558/1358062619.py:278: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tr_tot += float(loss) * bs


Ep 001 | tr=0.491665 (fcr=0.029238, shp=0.418570) | va=0.280404 (fcr=0.002888, shp=0.273184) | lr=1.00e-03 | best=0.280404 | bad=0/8
Ep 002 | tr=0.167462 (fcr=0.001989, shp=0.162489) | va=0.091938 (fcr=0.001298, shp=0.088692) | lr=1.00e-03 | best=0.091938 | bad=0/8
Ep 003 | tr=0.064156 (fcr=0.001244, shp=0.061045) | va=0.047856 (fcr=0.001042, shp=0.045251) | lr=1.00e-03 | best=0.047856 | bad=0/8
Ep 004 | tr=0.030786 (fcr=0.001067, shp=0.028119) | va=0.024675 (fcr=0.000799, shp=0.022678) | lr=1.00e-03 | best=0.024675 | bad=0/8
Ep 005 | tr=0.019486 (fcr=0.000918, shp=0.017191) | va=0.145320 (fcr=0.000819, shp=0.143272) | lr=1.00e-03 | best=0.024675 | bad=1/8
Ep 006 | tr=0.012877 (fcr=0.000840, shp=0.010776) | va=0.016907 (fcr=0.001393, shp=0.013426) | lr=1.00e-03 | best=0.016907 | bad=0/8
Ep 007 | tr=0.011029 (fcr=0.000729, shp=0.009207) | va=0.019092 (fcr=0.000645, shp=0.017480) | lr=1.00e-03 | best=0.016907 | bad=1/8
Ep 008 | tr=0.008797 (fcr=0.000666, shp=0.007133) | va=0.003696 (fcr=

In [ ]:
# ============================================================
# Export checkpoint, metrics, loss, scatter, ECDF, sample curves
# Academic CMU-style figures + ZIP download
# ============================================================

import os
import json
import time
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 0. Output directory
# ============================================================

OUT_DIR = Path("academic_outputs_LevelShapeParabolaNet")
FIG_DIR = OUT_DIR / "figures"
CKPT_DIR = OUT_DIR / "checkpoints"
TABLE_DIR = OUT_DIR / "tables"

for d in [OUT_DIR, FIG_DIR, CKPT_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. Academic Matplotlib style with CMU fallback
# ============================================================

def setup_academic_cmu_style():
    """
    Use CMU Serif if available.
    Fallback to Computer Modern / DejaVu Serif if CMU is unavailable.
    The universe will continue, barely.
    """
    available_fonts = {f.name for f in mpl.font_manager.fontManager.ttflist}

    if "CMU Serif" in available_fonts:
        main_font = "CMU Serif"
    elif "Computer Modern Roman" in available_fonts:
        main_font = "Computer Modern Roman"
    else:
        main_font = "DejaVu Serif"

    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": [main_font, "CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
        "mathtext.fontset": "cm",
        "axes.unicode_minus": False,

        "figure.figsize": (6.2, 4.6),
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.03,

        "axes.labelsize": 16,
        "axes.titlesize": 16,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 12,

        "axes.linewidth": 1.2,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 5,
        "ytick.major.size": 5,
        "xtick.minor.size": 3,
        "ytick.minor.size": 3,
        "xtick.major.width": 1.1,
        "ytick.major.width": 1.1,
        "xtick.minor.width": 0.9,
        "ytick.minor.width": 0.9,

        "legend.frameon": True,
        "legend.framealpha": 0.95,
        "legend.edgecolor": "0.2",

        "lines.linewidth": 2.0,
        "grid.linewidth": 0.7,
    })

    print(f"Using font: {main_font}")

setup_academic_cmu_style()


def boxed_axes(ax, grid=True):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.2)

    ax.tick_params(which="both", direction="in", top=True, right=True)

    if grid:
        ax.grid(True, which="major", alpha=0.35)
        ax.grid(True, which="minor", alpha=0.18)
        ax.minorticks_on()

    return ax


def savefig(fig, name):
    pdf_path = FIG_DIR / f"{name}.pdf"
    png_path = FIG_DIR / f"{name}.png"
    fig.savefig(pdf_path)
    fig.savefig(png_path)
    plt.close(fig)
    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")


# ============================================================
# 2. Save model checkpoint
# ============================================================

checkpoint = {
    "model_state_dict": model.state_dict(),
    "history": history,
    "W_GRID": W_GRID,
    "device": str(device),
    "training_time_seconds": float(ttrain) if "ttrain" in globals() else None,
    "model_class": model.__class__.__name__,
}

ckpt_path = CKPT_DIR / "best_model_state_dict.pt"
torch.save(checkpoint, ckpt_path)
print(f"Saved checkpoint: {ckpt_path}")


# ============================================================
# 3. Save training history
# ============================================================

history_df = pd.DataFrame(history)
history_csv_path = TABLE_DIR / "training_history.csv"
history_df.to_csv(history_csv_path, index=False)
print(f"Saved history: {history_csv_path}")


# ============================================================
# 4. Evaluate test set on original scale
# ============================================================

@torch.no_grad()
def collect_predictions_original_scale(model, dl, scF, scS, device="cpu"):
    model.eval()

    f_pred_s_list, f_true_s_list = [], []
    s_pred_s_list, s_true_s_list = [], []

    t0 = time.perf_counter()
    n_samples = 0

    is_cuda = str(device).startswith("cuda")

    if is_cuda:
        torch.cuda.synchronize()

    for xb, fb, sb in dl:
        xb = xb.to(device, non_blocking=True)
        fb = fb.to(device, non_blocking=True)
        sb = sb.to(device, non_blocking=True)

        fcr_p_s, shp_p_s, _ = model(xb)

        f_pred_s_list.append(fcr_p_s.detach().cpu().numpy())
        f_true_s_list.append(fb.detach().cpu().numpy())
        s_pred_s_list.append(shp_p_s.detach().cpu().numpy())
        s_true_s_list.append(sb.detach().cpu().numpy())

        n_samples += xb.size(0)

    if is_cuda:
        torch.cuda.synchronize()

    t1 = time.perf_counter()
    predict_time_s = t1 - t0

    f_pred_s = np.vstack(f_pred_s_list)
    f_true_s = np.vstack(f_true_s_list)
    s_pred_s = np.vstack(s_pred_s_list)
    s_true_s = np.vstack(s_true_s_list)

    f_pred = scF.inverse_transform(f_pred_s).reshape(-1, 1)
    f_true = scF.inverse_transform(f_true_s).reshape(-1, 1)

    s_pred = scS.inverse_transform(s_pred_s)
    s_true = scS.inverse_transform(s_true_s)

    W = W_GRID.reshape(1, -1)

    curve_pred = f_pred + s_pred[:, 0:1] * W + s_pred[:, 1:2] * (W ** 2)
    curve_true = f_true + s_true[:, 0:1] * W + s_true[:, 1:2] * (W ** 2)

    return {
        "f_pred": f_pred,
        "f_true": f_true,
        "s_pred": s_pred,
        "s_true": s_true,
        "curve_pred": curve_pred,
        "curve_true": curve_true,
        "predict_time_s": predict_time_s,
        "n_samples": n_samples,
    }


pred = collect_predictions_original_scale(
    model=model,
    dl=dl_te,
    scF=scF,
    scS=scS,
    device=device
)

f_pred = pred["f_pred"]
f_true = pred["f_true"]
s_pred = pred["s_pred"]
s_true = pred["s_true"]
curve_pred = pred["curve_pred"]
curve_true = pred["curve_true"]

predict_time_s = pred["predict_time_s"]
n_test = pred["n_samples"]

print(f"Prediction time: {predict_time_s:.6f} s")
print(f"Prediction speed: {n_test / predict_time_s:.2f} samples/s")


# ============================================================
# 5. Metrics
# ============================================================

def regression_metrics(y_true, y_pred, name):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    err = y_pred - y_true
    abs_err = np.abs(err)

    denom = np.maximum(np.abs(y_true), 1e-12)
    rel_err = abs_err / denom

    return {
        f"{name}_MAE": float(mean_absolute_error(y_true, y_pred)),
        f"{name}_RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{name}_R2": float(r2_score(y_true, y_pred)),
        f"{name}_mean_abs_error": float(np.mean(abs_err)),
        f"{name}_max_abs_error": float(np.max(abs_err)),
        f"{name}_mean_rel_error": float(np.mean(rel_err)),
        f"{name}_median_rel_error": float(np.median(rel_err)),
        f"{name}_p95_rel_error": float(np.percentile(rel_err, 95)),
        f"{name}_max_rel_error": float(np.max(rel_err)),
    }


metrics = {}

metrics.update(regression_metrics(f_true, f_pred, "Fcr"))
metrics.update(regression_metrics(s_true[:, 0], s_pred[:, 0], "Lambda1"))
metrics.update(regression_metrics(s_true[:, 1], s_pred[:, 1], "Lambda2"))

curve_rmse_per_sample = np.sqrt(np.mean((curve_pred - curve_true) ** 2, axis=1))
curve_rel_l2_per_sample = (
    np.linalg.norm(curve_pred - curve_true, axis=1)
    / np.maximum(np.linalg.norm(curve_true, axis=1), 1e-12)
)

metrics.update({
    "Curve_RMSE_mean": float(np.mean(curve_rmse_per_sample)),
    "Curve_RMSE_median": float(np.median(curve_rmse_per_sample)),
    "Curve_RMSE_p95": float(np.percentile(curve_rmse_per_sample, 95)),
    "Curve_RMSE_max": float(np.max(curve_rmse_per_sample)),

    "Curve_rel_L2_mean": float(np.mean(curve_rel_l2_per_sample)),
    "Curve_rel_L2_median": float(np.median(curve_rel_l2_per_sample)),
    "Curve_rel_L2_p95": float(np.percentile(curve_rel_l2_per_sample, 95)),
    "Curve_rel_L2_max": float(np.max(curve_rel_l2_per_sample)),

    "n_test": int(n_test),
    "training_time_seconds": float(ttrain) if "ttrain" in globals() else None,
    "prediction_time_seconds": float(predict_time_s),
    "prediction_ms_per_sample": float(1e3 * predict_time_s / max(n_test, 1)),
    "prediction_samples_per_second": float(n_test / max(predict_time_s, 1e-12)),
    "device": str(device),
})

metrics_json_path = TABLE_DIR / "metrics_summary.json"
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

metrics_df = pd.DataFrame([metrics])
metrics_csv_path = TABLE_DIR / "metrics_summary.csv"
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"Saved metrics JSON: {metrics_json_path}")
print(f"Saved metrics CSV : {metrics_csv_path}")

display(metrics_df.T.rename(columns={0: "value"}))


# ============================================================
# 6. Save raw prediction table
# ============================================================

pred_df = pd.DataFrame({
    "Fcr_true": f_true.reshape(-1),
    "Fcr_pred": f_pred.reshape(-1),
    "Lambda1_true": s_true[:, 0],
    "Lambda1_pred": s_pred[:, 0],
    "Lambda2_true": s_true[:, 1],
    "Lambda2_pred": s_pred[:, 1],
    "Curve_RMSE": curve_rmse_per_sample,
    "Curve_rel_L2": curve_rel_l2_per_sample,
})

pred_csv_path = TABLE_DIR / "test_predictions_summary.csv"
pred_df.to_csv(pred_csv_path, index=False)
print(f"Saved predictions: {pred_csv_path}")


# ============================================================
# 7. Plot training and validation losses
# ============================================================

fig, ax = plt.subplots(figsize=(6.4, 4.6))

ax.plot(history_df["epoch"], history_df["tr_total"], label="Training")
ax.plot(history_df["epoch"], history_df["va_total"], label="Validation")

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Total loss")
ax.set_title("Training history")
ax.legend()
boxed_axes(ax)

savefig(fig, "fig01_training_validation_loss")


fig, ax = plt.subplots(figsize=(6.4, 4.6))

ax.plot(history_df["epoch"], history_df["tr_fcr"], label=r"Train $F_{\mathrm{cr}}$")
ax.plot(history_df["epoch"], history_df["va_fcr"], label=r"Val. $F_{\mathrm{cr}}$")
ax.plot(history_df["epoch"], history_df["tr_shp"], label=r"Train shape")
ax.plot(history_df["epoch"], history_df["va_shp"], label=r"Val. shape")

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss component")
ax.set_title("Loss components")
ax.legend(ncol=1)
boxed_axes(ax)

savefig(fig, "fig02_loss_components")


# ============================================================
# 8. Scatter / parity plots
# ============================================================

def parity_plot(y_true, y_pred, xlabel, ylabel, title, filename, max_points=60000):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    n = len(y_true)
    if n > max_points:
        rng = np.random.default_rng(42)
        idx = rng.choice(n, size=max_points, replace=False)
        yt = y_true[idx]
        yp = y_pred[idx]
    else:
        yt = y_true
        yp = y_pred

    lo = min(np.min(yt), np.min(yp))
    hi = max(np.max(yt), np.max(yp))
    pad = 0.04 * (hi - lo + 1e-12)
    lo -= pad
    hi += pad

    fig, ax = plt.subplots(figsize=(5.4, 5.2))

    ax.scatter(yt, yp, s=5, alpha=0.35, edgecolors="none")
    ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.6, label="Ideal")

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="best")
    boxed_axes(ax)

    savefig(fig, filename)


parity_plot(
    f_true, f_pred,
    xlabel=r"True $F_{\mathrm{cr}}$",
    ylabel=r"Predicted $F_{\mathrm{cr}}$",
    title=r"Parity plot for $F_{\mathrm{cr}}$",
    filename="fig03_parity_Fcr"
)

parity_plot(
    s_true[:, 0], s_pred[:, 0],
    xlabel=r"True $\Lambda_1$",
    ylabel=r"Predicted $\Lambda_1$",
    title=r"Parity plot for $\Lambda_1$",
    filename="fig04_parity_Lambda1"
)

parity_plot(
    s_true[:, 1], s_pred[:, 1],
    xlabel=r"True $\Lambda_2$",
    ylabel=r"Predicted $\Lambda_2$",
    title=r"Parity plot for $\Lambda_2$",
    filename="fig05_parity_Lambda2"
)


# ============================================================
# 9. ECDF plots
# ============================================================

def ecdf_plot(values, xlabel, title, filename):
    values = np.asarray(values).reshape(-1)
    values = values[np.isfinite(values)]
    values = np.sort(values)
    y = np.arange(1, len(values) + 1) / len(values)

    fig, ax = plt.subplots(figsize=(6.2, 4.6))

    ax.plot(values, y)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Empirical cumulative probability")
    ax.set_title(title)
    boxed_axes(ax)

    savefig(fig, filename)


rel_fcr = np.abs(f_pred.reshape(-1) - f_true.reshape(-1)) / np.maximum(np.abs(f_true.reshape(-1)), 1e-12)
rel_l1 = np.abs(s_pred[:, 0] - s_true[:, 0]) / np.maximum(np.abs(s_true[:, 0]), 1e-12)
rel_l2 = np.abs(s_pred[:, 1] - s_true[:, 1]) / np.maximum(np.abs(s_true[:, 1]), 1e-12)

ecdf_plot(
    rel_fcr,
    xlabel=r"Relative error of $F_{\mathrm{cr}}$",
    title=r"ECDF of $F_{\mathrm{cr}}$ relative error",
    filename="fig06_ecdf_Fcr_relative_error"
)

ecdf_plot(
    curve_rel_l2_per_sample,
    xlabel=r"Relative $L_2$ error of reconstructed curve",
    title=r"ECDF of curve relative $L_2$ error",
    filename="fig07_ecdf_curve_relative_L2"
)


# Combined ECDF
fig, ax = plt.subplots(figsize=(6.4, 4.8))

for values, label in [
    (rel_fcr, r"$F_{\mathrm{cr}}$"),
    (rel_l1, r"$\Lambda_1$"),
    (rel_l2, r"$\Lambda_2$"),
    (curve_rel_l2_per_sample, r"Curve $L_2$"),
]:
    values = np.sort(np.asarray(values).reshape(-1))
    y = np.arange(1, len(values) + 1) / len(values)
    ax.plot(values, y, label=label)

ax.set_xlabel("Relative error")
ax.set_ylabel("Empirical cumulative probability")
ax.set_title("ECDF of prediction errors")
ax.legend()
boxed_axes(ax)

savefig(fig, "fig08_ecdf_combined_errors")


# ============================================================
# 10. Representative reconstructed curves
# ============================================================

def pick_representative_indices(error_array, n_each=2):
    """
    Select representative cases:
    - best
    - median
    - near 75th percentile
    - near 95th percentile
    - worst
    """
    e = np.asarray(error_array).reshape(-1)

    targets = [
        np.min(e),
        np.percentile(e, 50),
        np.percentile(e, 75),
        np.percentile(e, 95),
        np.max(e),
    ]

    indices = []
    for t in targets:
        idx = int(np.argmin(np.abs(e - t)))
        indices.append(idx)

    # remove duplicates while preserving order
    seen = set()
    unique = []
    for idx in indices:
        if idx not in seen:
            unique.append(idx)
            seen.add(idx)

    return unique


sample_indices = pick_representative_indices(curve_rel_l2_per_sample)

W = W_GRID.reshape(-1)

fig, ax = plt.subplots(figsize=(7.0, 5.0))

for k, idx in enumerate(sample_indices):
    ax.plot(
        W,
        curve_true[idx],
        linestyle="-",
        linewidth=2.0,
        label=rf"True case {k+1}"
    )
    ax.plot(
        W,
        curve_pred[idx],
        linestyle="--",
        linewidth=1.8,
        label=rf"Pred. case {k+1}"
    )

ax.set_xlabel(r"Normalized transverse coordinate $W$")
ax.set_ylabel(r"Load parameter $\lambda(W)$")
ax.set_title("Representative reconstructed curves")
ax.legend(ncol=2, fontsize=10)
boxed_axes(ax)

savefig(fig, "fig09_representative_curves")


# Individual sample figures
for k, idx in enumerate(sample_indices):
    fig, ax = plt.subplots(figsize=(6.2, 4.6))

    ax.plot(W, curve_true[idx], linestyle="-", linewidth=2.3, label="Ground truth")
    ax.plot(W, curve_pred[idx], linestyle="--", linewidth=2.0, label="Prediction")

    ax.set_xlabel(r"Normalized transverse coordinate $W$")
    ax.set_ylabel(r"Load parameter $\lambda(W)$")
    ax.set_title(
        rf"Representative case {k+1}: "
        rf"rel. $L_2$ = {curve_rel_l2_per_sample[idx]:.3e}"
    )

    ax.legend()
    boxed_axes(ax)

    savefig(fig, f"fig10_sample_curve_case_{k+1:02d}")


# ============================================================
# 11. Runtime bar plot
# ============================================================

training_time = float(ttrain) if "ttrain" in globals() else np.nan
prediction_time = float(predict_time_s)

runtime_df = pd.DataFrame({
    "Quantity": ["Training time", "Full test prediction time", "Prediction ms/sample"],
    "Value": [
        training_time,
        prediction_time,
        1e3 * prediction_time / max(n_test, 1),
    ],
    "Unit": ["s", "s", "ms/sample"],
})

runtime_csv_path = TABLE_DIR / "runtime_summary.csv"
runtime_df.to_csv(runtime_csv_path, index=False)
print(f"Saved runtime summary: {runtime_csv_path}")

fig, ax = plt.subplots(figsize=(5.8, 4.4))

x = np.arange(2)
vals = [training_time, prediction_time]
labels = ["Training", "Prediction"]

ax.bar(x, vals)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Time (s)")
ax.set_title("Computational runtime")
boxed_axes(ax)

for i, v in enumerate(vals):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=12)

savefig(fig, "fig11_runtime_summary")


# ============================================================
# 12. Save compact report text
# ============================================================

report_path = OUT_DIR / "report_summary.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("LevelShapeParabolaNet evaluation report\n")
    f.write("=" * 70 + "\n")
    f.write(f"Device: {device}\n")
    f.write(f"Number of test samples: {n_test:,}\n")
    f.write(f"Training time [s]: {training_time:.6f}\n")
    f.write(f"Prediction time [s]: {prediction_time:.6f}\n")
    f.write(f"Prediction ms/sample: {1e3 * prediction_time / max(n_test, 1):.9f}\n")
    f.write(f"Prediction samples/s: {n_test / max(prediction_time, 1e-12):.6f}\n")
    f.write("\nMetrics:\n")
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")

print(f"Saved report: {report_path}")


# ============================================================
# 13. ZIP all outputs and download
# ============================================================

zip_path = shutil.make_archive(
    base_name=str(OUT_DIR),
    format="zip",
    root_dir=str(OUT_DIR)
)

print("=" * 70)
print(f"ZIP saved to: {zip_path}")
print("=" * 70)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Not running in Colab or download unavailable.")
    print("ZIP path:", zip_path)
    print("Exception:", repr(e))

Using font: DejaVu Serif
Saved checkpoint: academic_outputs_LevelShapeParabolaNet/checkpoints/best_model_state_dict.pt
Saved history: academic_outputs_LevelShapeParabolaNet/tables/training_history.csv
Prediction time: 6.004849 s
Prediction speed: 78061.91 samples/s
Saved metrics JSON: academic_outputs_LevelShapeParabolaNet/tables/metrics_summary.json
Saved metrics CSV : academic_outputs_LevelShapeParabolaNet/tables/metrics_summary.csv


,value
Fcr_MAE,0.00715
Fcr_RMSE,0.009111
Fcr_R2,0.999942
Fcr_mean_abs_error,0.00715
Fcr_max_abs_error,0.056068
Fcr_mean_rel_error,0.002706
Fcr_median_rel_error,0.002189
Fcr_p95_rel_error,0.006969
Fcr_max_rel_error,0.026133
Lambda1_MAE,0.0


Saved predictions: academic_outputs_LevelShapeParabolaNet/tables/test_predictions_summary.csv
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig01_training_validation_loss.pdf
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig01_training_validation_loss.png
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig02_loss_components.pdf
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig02_loss_components.png
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig03_parity_Fcr.pdf
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig03_parity_Fcr.png
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig04_parity_Lambda1.pdf
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig04_parity_Lambda1.png
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig05_parity_Lambda2.pdf
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig05_parity_Lambda2.png
Saved: academic_outputs_LevelShapeParabolaNet/figures/fig06_ecdf_Fcr_relative_error.pdf
Saved: academi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SAVE EVERYTHING: model + scalers + config + metrics + history
# ============================================================

import os
import json
import joblib
from datetime import datetime

# Đổi tên folder cho đỡ nhầm giữa các model
SAVE_DIR = "saved_LevelShapeParabolaNet_GELU_5levels_DT123"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Save model checkpoint
# ------------------------------------------------------------
checkpoint = {
    # Model state
    "model_state_dict": model.state_dict(),

    # Architecture
    "model_class": "LevelShapeParabolaNet",
    "ENC": ENC,
    "ACT": ACT,
    "DROPOUT": DROPOUT,
    "BATCHNORM": BATCHNORM,

    # Input/output definition
    "VAR_COLS": VAR_COLS,
    "FCR_COL": FCR_COL,
    "L1_COL": L1_COL,
    "L2_COL": L2_COL,
    "L0_COL": L0_COL,
    "Nw": Nw,
    "W_GRID": W_GRID,

    # Loss setup
    "LAMBDA_FCR": LAMBDA_FCR,
    "LAMBDA_SHAPE": LAMBDA_SHAPE,
    "USE_CURVE_LOSS": USE_CURVE_LOSS,
    "LAMBDA_CURVE": LAMBDA_CURVE,

    # Training setup
    "EPOCHS": EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "LR_FACTOR": LR_FACTOR,
    "LR_PATIENCE": LR_PATIENCE,
    "LR_MIN": LR_MIN,
    "ES_PATIENCE": ES_PATIENCE,
    "ES_MIN_DELTA": ES_MIN_DELTA,
    "USE_AMP": USE_AMP,

    # Dataset info
    "CSV_PATH": CSV_PATH,
    "csv_shape": df.shape,
    "csv_columns": df.columns.tolist(),
    "n_total": len(df),
    "n_train": len(X_tr_s),
    "n_val": len(X_va_s),
    "n_test": len(X_te_s),

    # Parameter count
    "params_total": p_all,
    "params_trainable": p_tr,

    # Timing
    "train_time_seconds": ttrain,
    "test_predict_total_seconds": total_s,
    "test_predict_ms_per_sample": ms_s,
    "test_predict_samples_per_second": sps,
    "test_predict_n_samples": n,

    # Metrics in original scale
    "val_RMSE_Fcr": rmse_f_val,
    "val_RMSE_L1L2": rmse_s_val,
    "val_RMSE_Curve50": rmse_c_val,
    "test_RMSE_Fcr": rmse_f_te,
    "test_RMSE_L1L2": rmse_s_te,
    "test_RMSE_Curve50": rmse_c_te,

    # Misc
    "device": device,
    "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

torch.save(checkpoint, os.path.join(SAVE_DIR, "model_checkpoint.pt"))

# ------------------------------------------------------------
# 2. Save scalers
# ------------------------------------------------------------
joblib.dump(scX, os.path.join(SAVE_DIR, "scX.pkl"))
joblib.dump(scF, os.path.join(SAVE_DIR, "scF.pkl"))
joblib.dump(scS, os.path.join(SAVE_DIR, "scS.pkl"))

# ------------------------------------------------------------
# 3. Save training history
# ------------------------------------------------------------
history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)

# ------------------------------------------------------------
# 4. Save metrics/config as readable JSON
# ------------------------------------------------------------
metadata = checkpoint.copy()

# Convert objects not directly JSON serializable
metadata["W_GRID"] = W_GRID.tolist()
metadata["csv_shape"] = list(df.shape)

# Remove actual tensor weights from JSON metadata
metadata.pop("model_state_dict", None)

with open(os.path.join(SAVE_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# 5. Save prediction report as CSV
# ------------------------------------------------------------
report = {
    "Model": ["LevelShapeParabolaNet"],
    "Activation": [ACT],
    "Encoder": [str(ENC)],
    "Total parameters": [p_all],
    "Trainable parameters": [p_tr],
    "Train time (s)": [ttrain],
    "Full-test predict time (s)": [total_s],
    "Predict ms/sample": [ms_s],
    "Predict samples/s": [sps],
    "VAL RMSE Fcr": [rmse_f_val],
    "VAL RMSE L1L2": [rmse_s_val],
    "VAL RMSE Curve50": [rmse_c_val],
    "TEST RMSE Fcr": [rmse_f_te],
    "TEST RMSE L1L2": [rmse_s_te],
    "TEST RMSE Curve50": [rmse_c_te],
    "CSV": [CSV_PATH],
    "n_train": [len(X_tr_s)],
    "n_val": [len(X_va_s)],
    "n_test": [len(X_te_s)],
}

report_df = pd.DataFrame(report)
report_df.to_csv(os.path.join(SAVE_DIR, "model_report.csv"), index=False)

print("=" * 80)
print("Saved everything successfully.")
print("SAVE_DIR:", SAVE_DIR)
print("- model checkpoint :", os.path.join(SAVE_DIR, "model_checkpoint.pt"))
print("- input scaler     :", os.path.join(SAVE_DIR, "scX.pkl"))
print("- Fcr scaler       :", os.path.join(SAVE_DIR, "scF.pkl"))
print("- shape scaler     :", os.path.join(SAVE_DIR, "scS.pkl"))
print("- history          :", os.path.join(SAVE_DIR, "training_history.csv"))
print("- metadata         :", os.path.join(SAVE_DIR, "metadata.json"))
print("- report           :", os.path.join(SAVE_DIR, "model_report.csv"))
print("=" * 80)

Saved everything successfully.
SAVE_DIR: saved_LevelShapeParabolaNet_GELU_5levels_DT123
- model checkpoint : saved_LevelShapeParabolaNet_GELU_5levels_DT123/model_checkpoint.pt
- input scaler     : saved_LevelShapeParabolaNet_GELU_5levels_DT123/scX.pkl
- Fcr scaler       : saved_LevelShapeParabolaNet_GELU_5levels_DT123/scF.pkl
- shape scaler     : saved_LevelShapeParabolaNet_GELU_5levels_DT123/scS.pkl
- history          : saved_LevelShapeParabolaNet_GELU_5levels_DT123/training_history.csv
- metadata         : saved_LevelShapeParabolaNet_GELU_5levels_DT123/metadata.json
- report           : saved_LevelShapeParabolaNet_GELU_5levels_DT123/model_report.csv


In [ ]:
# ============================================================
# ZIP + DOWNLOAD EVERYTHING
# ============================================================

import os
import shutil
from google.colab import files

# Folder đã save ở cell trước
# SAVE_DIR = "saved_LevelShapeParabolaNet_GELU_5levels_DT123"

zip_base = SAVE_DIR.rstrip("/\\")
zip_path = f"{zip_base}.zip"

# Nếu file zip cũ tồn tại thì xóa để tránh nhầm
if os.path.exists(zip_path):
    os.remove(zip_path)

# Nén toàn bộ folder SAVE_DIR
shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=SAVE_DIR
)

print("=" * 80)
print("ZIP created successfully.")
print("ZIP path:", zip_path)
print("ZIP size [MB]:", os.path.getsize(zip_path) / 1024**2)
print("=" * 80)

# Tải về máy
files.download(zip_path)

ZIP created successfully.
ZIP path: saved_LevelShapeParabolaNet_GELU_5levels_DT123.zip
ZIP size [MB]: 0.4896087646484375


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>